In [41]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')
sys.path.insert(1, r'C:\Users\tomaz.bregar\Desktop\git_projects\pyEMA')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import pyFBS #v __init.py__ bi morale knjižnice IO, SEMM in VPT pisati z malo: io, semm, vpt?
from pyEMA import pyEMA
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from time import time,sleep
from pyFBS.utility import *

## Example of modal analysis on a simple beam

#### 3D View
Open 3Dviewer in the background, with CSYS in origin

In [61]:
view3D = pyFBS.display.view3D()

Add a structure from .stl file to the 3D 

In [62]:
AB_stl = "../data/LADISK_laser/STL/T_structure.stl"
view3D.add_stl(AB_stl,color = "#8FB1CC",opacity = 0.5)

#### Accelerometers
Add accelerometers from .xlsx file together with appropriate labels

In [45]:
AB = "../data/LADISK_laser/Measurements/Excel/T_structure.xlsx"
df = pd.read_excel(AB, sheet_name='Sensors_A')
#view3D.show_acc(df)
#df
#view3D.label_acc(df)

#### Channels
Add corresponding channels from .xlsx file together with appropriate labels

In [46]:
df = pd.read_excel(AB, sheet_name='Channels_A')

#view3D.show_chn(df)
#view3D.label_chn(df)
#df

#### Impacts
Add impacts from .xlsx with appropriate labels

In [47]:
df = pd.read_excel(AB, sheet_name='Impacts_A')

#view3D.show_imp(df)
#view3D.label_imp(df)
#df

### Modal analysis
First load the data

In [48]:
freq = pd.read_csv("../data/LADISK_laser/Measurements/freq.txt", sep='	', squeeze=True, header=None).to_numpy()
Y = pd.read_csv("../data/LADISK_laser/Measurements/FRF.txt", sep='	', dtype=object, header=None,)\
            .applymap(lambda s: np.complex(s.replace('i', 'j'))).to_numpy()

Modeshape identification using pyEMA (LSCF/LSFD)

In [54]:
a = pyEMA.Model(
    Y,
    freq,
    pol_order_high=30,
    lower = 500,
    upper = 3200
    )

a.get_poles()

#fn_temp, xi_temp, test_fn, test_xi = pyEMA.stabilisation(a.all_poles, a.pol_order_high, err_fn=1, err_xi=0.5)

approx_nat_freq = [938, 1973, 2150, 2678, 2751, 3024]
a.select_closest_poles(approx_nat_freq)
#a.stab_chart()

100%|██████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 249.22it/s]


In [56]:
#tukaj preoblikujem matriko Y sicer naslednja celica ne deluje - če jo preoblikujem prej ne deluje pyEMA
Y = np.reshape(Y, (63, 1, 3200))

In [57]:
H_acc, A_acc = a.get_constants(whose_poles=a,least_squares_type="old")
_modes_acc = modeshape_scaling_DP(A_acc,0)

new_mode = unflattenFRFs(_modes_acc,Y)

#### Create mesh

In [58]:
_newnew_mode = np.zeros((63,3,6),dtype = complex)
_newnew_mode[:,2:3,:] = new_mode

In [63]:
df = pd.read_excel(AB, sheet_name='Sensors_A')
pos_array = df[["Position_1","Position_2","Position_3"]].to_numpy()*1000

point_cloud = pv.PolyData(pos_array)
surf = point_cloud.delaunay_2d()
mesh_actor = view3D.plot.add_mesh(surf,scalars = np.zeros(np.shape(df[["Position_1","Position_2","Position_3"]].to_numpy())[0]),clim = [-1,1],color = "k", cmap="viridis",name = "mesh")#,show_edges=True)#,render_points_as_spheres = True,point_size=20)

pts = point_cloud.points.copy()

#### Display modeshape and add Animate button

In [64]:
select = 5
scale = 10

mode_shape = _newnew_mode[:,:,select]#@R 
#complex_plot_3D(mode_shape)
mode_dict = dict()

mode_dict["freq"] = a.nat_freq[select]
mode_dict["damp"] = a.nat_xi[select]*100
mode_dict["mcf"] = MCF(mode_shape.flatten())*100

mode_dict["animation_pts"] = mode_animation(mode_shape,scale,no_points = 60)
mode_dict["mesh"] = surf
mode_dict["or_pts"] = pts

mode_dict["scalars"] = True
mode_dict["fps"] = 30

view3D.add_modeshape(mode_dict,run_animation = True)